# 04 — Job/Role Clustering

This notebook trains a K-Means model on the job corpus, determines cluster topics/labels, and saves a 2D PCA visualization.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
from src import config
from src.features.text_features import build_tfidf_vectorizer
from src.models.clustering import JobClustering

In [ ]:
# Load clean jobs data
jobs_df = pd.read_csv(config.JOBS_CLEAN_CSV)
jobs_df['text'] = jobs_df['text'].fillna('')
print(f"Loaded {len(jobs_df):,} job descriptions.")

In [ ]:
# Vectorize job corpus
print("Vectorizing jobs...")
vectorizer = build_tfidf_vectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(jobs_df['text'])
print(f"TF-IDF shape: {tfidf_matrix.shape}")

In [ ]:
# Run clustering
print("Training K-Means...")
clustering = JobClustering(n_clusters=8)
clustering.fit(tfidf_matrix)

labels = clustering.predict(tfidf_matrix)
jobs_df['cluster'] = labels

# Output top terms and labels
cluster_labels = clustering.label_clusters(vectorizer)
for cid, label in cluster_labels.items():
    print(f"Cluster {cid}: {label}")

In [ ]:
# Generate and save PCA plot
print("Generating PCA projection plot...")
config.PROJECT_ROOT.joinpath('reports/figures').mkdir(parents=True, exist_ok=True)
output_path = config.PROJECT_ROOT / 'reports/figures/job_clusters.png'
clustering.plot_clusters(tfidf_matrix, labels, output_path)
print(f"Cluster plot saved to: {output_path}")